# Gộp 5 dạng: D04, D14, D28, D32, D33

Chạy **FULL 90 câu** bằng **ReAct + Calculator tool** (Qwen3-4B), **nạp model
DUY NHẤT 1 LẦN** cho cả 5 dạng, chạy tuần tự từng dạng, xuất
1 file CSV riêng cho mỗi dạng. Backend `MayTinh` (sympy, chính xác tuyệt đối,
có bộ nhớ biến) là DUY NHẤT, dùng chung cho mọi dạng (đã xác nhận giống hệt
nhau giữa tất cả các dạng bằng hash). Prompt (PART_TOOL/PART_KIENTHUC/
PART_HUONGGIAI/PART_FEWSHOT/PART_NHIEMVU) của MỖI dạng được trích **nguyên
văn** từ notebook 1 câu riêng của dạng đó (`KLTN_{MA_DANG}_ReAct_Calculator_1cau.ipynb`),
đã kiểm chứng độc lập khớp 91/91 với sympy + 90/90 với backend thật trước khi
đưa vào đây.

Mỗi dạng chạy vòng lặp ReAct THEO LÔ (batch), qua hàm dùng chung
`chay_mot_dang(...)`: mỗi vòng gọi vLLM một lần cho tất cả câu đang hoạt động
của dạng đó, rồi chạy Calculator riêng cho từng câu, rồi generate tiếp; mỗi
câu có bộ nhớ biến riêng, ngân sách token riêng, tự thoát khi viết xong
`Final Answer`.

## Trước khi chạy

Upload các file `plan_solve_prompts_{MA_DANG}.json` (90 câu mỗi dạng, đã loại
câu gốc nằm trong few-shot) thành Kaggle Dataset, gắn vào notebook, bật GPU.

Danh sách dạng và file dữ liệu cần có:
- D04: `/kaggle/input/.../plan_solve_prompts_D04.json` -> `/kaggle/working/d04_react_calculator_full90.csv`
- D14: `/kaggle/input/.../plan_solve_prompts_D14.json` -> `/kaggle/working/d14_react_calculator_full90.csv`
- D28: `/kaggle/input/.../plan_solve_prompts_D28.json` -> `/kaggle/working/d28_react_calculator_full90.csv`
- D32: `/kaggle/input/.../plan_solve_prompts_D32.json` -> `/kaggle/working/d32_react_calculator_full90.csv`
- D33: `/kaggle/input/.../plan_solve_prompts_D33.json` -> `/kaggle/working/d33_react_calculator_full90.csv`

(Sửa `DATA_DIR` ở Cell cấu hình nếu đường dẫn Kaggle Dataset khác.)

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_DIR = '/kaggle/input/soict-11dang-full90'  # SUA NEU KHAC ten Kaggle Dataset
OUT_DIR  = '/kaggle/working'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau (dung chung moi dang).
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ---- Backend Calculator: DUY NHAT, dung chung cho MOI dang ----
# (dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao - an toan voi
# multiprocessing.fork, xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang - model nap 1 lan, dung chung cho ca 5 dang.')

In [ ]:
# =====================================================================
# HAM DUNG CHUNG: chay vong lap ReAct+Calculator THEO LO cho 1 dang
# (goi lai nhieu lan ben duoi, moi lan 1 dang, dung chung llm/tok/MayTinh
# da nap o Cell truoc - khong nap lai model)
# =====================================================================
STOP_STR = 'Observation:'
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt, an_so_tu_do):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=an_so_tu_do)   # bo nho + an so RIENG cho tung cau
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


def chay_mot_dang(ma_dang, prompt_template, an_so_tu_do, max_tool_calls,
                   data_path, out_path):
    print('=' * 72)
    print(f'BAT DAU DANG {ma_dang} (MAX_TOOL_CALLS={max_tool_calls})')
    print('=' * 72)

    with open(data_path, encoding='utf-8') as f:
        records = json.load(f)
    print('So cau:', len(records))

    prompts_ban_dau = []
    for r in records:
        prompt_text = prompt_template.replace('{de_bai}', r['de_bai_mcq'])
        prompts_ban_dau.append(tok.apply_chat_template(
            [{'role': 'user', 'content': prompt_text}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True))

    ds = [TrangThai(r, p, an_so_tu_do) for r, p in zip(records, prompts_ban_dau)]

    t0 = time.time()
    vong = 0
    while True:
        hoat_dong = [s for s in ds if not s.xong]
        if not hoat_dong:
            break
        vong += 1

        lo = []
        for s in hoat_dong:
            dau_vao = s.chat_prompt + s.full_text
            cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
            so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
            if so_sinh <= 0:
                s.xong = True
                s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
                continue
            stop = None if s.buoc_chot else [STOP_STR]
            lo.append((s, dau_vao, SamplingParams(
                temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
                seed=SEED, stop=stop)))

        if not lo:
            break

        outs = llm.generate([p for _, p, _ in lo],
                            [sp for _, _, sp in lo], use_tqdm=False)

        n_goi_vong_nay = 0
        for (s, _, _), out in zip(lo, outs):
            o = out.outputs[0]
            s.full_text += o.text
            s.tong_tok += len(o.token_ids)

            if s.buoc_chot:
                s.xong = True
                s.ly_do_dung = 'het_han_muc_tool'
                continue

            m_final = FINAL_ANSWER_RE.search(s.full_text)
            if m_final:
                s.full_text = s.full_text[:m_final.end()]
                s.xong = True
                s.ly_do_dung = 'model_ket_thuc'
                continue

            if o.stop_reason != STOP_STR:
                s.xong = True
                s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                                else 'het_token')
                continue

            s.n_calls += 1
            n_goi_vong_nay += 1
            cac_khop = list(ACTION_INPUT_RE.finditer(o.text))
            khop = cac_khop[0] if cac_khop else None
            bieu_thuc = khop.group(1).strip() if khop else ''
            if len(cac_khop) > 1:
                vi_tri_cat = len(s.full_text) - len(o.text) + khop.end()
                s.full_text = s.full_text[:vi_tri_cat]

            if not bieu_thuc:
                quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                            'then "Action: Calculator", then "Action Input: <expression>".')
                s.nhat_ky.append(('(khong co Action Input)', quan_sat))
            else:
                ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
                if loi:
                    quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                                '(use I for the imaginary unit, sqrt() for roots, '
                                '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                    s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
                else:
                    quan_sat = ket_qua
                    s.nhat_ky.append((bieu_thuc, ket_qua))

            s.full_text += f'{STOP_STR} {quan_sat}\n'

            if s.n_calls >= max_tool_calls:
                s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                                'Answer now, with no further Action.]\n')
                s.buoc_chot = True

        print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
              f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
              f'({time.time()-t0:.0f}s)')

    print(f'Xong {ma_dang} sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')

    rows = []
    for s in ds:
        chon = lay_dap_an(s.full_text)
        n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
        rows.append({
            'STT': s.rec['STT'],
            'loai_so': s.rec.get('loai_so', ''),
            'dap_an_dung': s.rec['dap_an_letter'],
            'model_chon': chon,
            'DUNG': chon == s.rec['dap_an_letter'],
            'token': s.tong_tok,
            'so_luot_goi_tool': s.n_calls,
            'so_loi_cu_phap': n_loi,
            'ly_do_dung': s.ly_do_dung,
            'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
            'full_text': s.full_text,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    print('-' * 72)
    print(f"KET QUA {ma_dang}: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
    print('-' * 72)
    print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
    print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
          f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
    print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
    print(df['ly_do_dung'].value_counts())
    if (~df['DUNG']).any():
        print('--- CAC CAU SAI ---')
        print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                               'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                               'ly_do_dung']].to_string(index=False))
    else:
        print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
    print('Da luu:', out_path)
    print()
    return df


In [ ]:
# =====================================================================
# DANG D04 (prompt trich nguyen van tu KLTN_D04_ReAct_Calculator_1cau.ipynb)
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- Real part: `re(x)`. Imaginary part: `im(x)` (this already returns just the coefficient, never multiplied by `I`).
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Finding the conjugate's algebraic form.** The problem always gives a complex number $z=a+bi$ ($a,b$ real) and asks for the real and imaginary parts of its conjugate $\overline{z}$. By fact 3, the conjugate is formed by keeping the real part exactly as it is and flipping the sign of the imaginary part: $\overline{z}=a-bi$. This is the entire computation; there is no equation to solve here, only a direct application of the definition to whatever $a$ and $b$ the problem states.

**Reading off the real and imaginary parts, and the classic trap.** For any complex number written in the standard form $A+Bi$ ($A,B$ real), $\mathrm{Re}=A$ and $\mathrm{Im}=B$ (fact 9). Applied to $\overline{z}=a-bi$: the real part is $a$ (unchanged from $z$), and the imaginary part is $-b$ (the sign of $z$'s own imaginary part, flipped). The imaginary part is ALWAYS a bare real number, by definition; it never carries the symbol $i$ attached to it. This is exactly where multiple-choice options try to trip a student up: an option that states "phần ảo bằng $-bi$" (with an $i$ still attached) is wrong by definition alone, regardless of whether $-b$ is numerically the right number; a genuine imaginary part is a number, never a number-times-$i$. Reading which options carry this $i$-suffix trap, separately from checking whether the two numbers themselves are correct, is the last step.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Cho số phức $z = 3-2i$. Tìm phần thực và phần ảo của số phức $\overline{z}$.

A. Phần thực bằng $-3$ và phần ảo bằng $-2i$.   B. Phần thực bằng $-3$ và phần ảo bằng $-2$.   C. Phần thực bằng $3$ và phần ảo bằng $2i$.   D. Phần thực bằng $3$ và phần ảo bằng $2$.

<think>
Thought: The problem states $z=3-2i$, quoted verbatim. I store it.
Action: Calculator
Action Input: z = 3-2*I
Observation: z = 3 - 2*I

Thought: By PART 2, $\overline{z}$ keeps the real part and flips the sign of the imaginary part; I compute it with the Calculator rather than doing this flip in my head, since even a sign flip is exactly the kind of "obvious" step where a mistake can slip in unnoticed.
Action: Calculator
Action Input: conj_z = conjugate(z)
Observation: conj_z = 3 + 2*I

Thought: Now I read off the real part of `conj_z`.
Action: Calculator
Action Input: re_val = re(conj_z)
Observation: re_val = 3

Thought: And the imaginary part of `conj_z`.
Action: Calculator
Action Input: im_val = im(conj_z)
Observation: im_val = 2

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "Phần thực bằng $-3$ và phần ảo bằng $-2i$."; B: "Phần thực bằng $-3$ và phần ảo bằng $-2$."; C: "Phần thực bằng $3$ và phần ảo bằng $2i$."; D: "Phần thực bằng $3$ và phần ảo bằng $2$." A genuine imaginary part is always a bare number, never carrying $i$: option A's stated imaginary part is "$-2i$", which carries $i$, so A is wrong by this format rule alone, with no Calculator needed. Likewise option C's stated imaginary part is "$2i$", also carrying $i$, so C is wrong the same way. That leaves B and D as the only options whose format could possibly be right; I test both of their two numbers. Starting with B's real part, $-3$.
Action: Calculator
Action Input: test_B_re = Eq(re_val, -3)
Observation: test_B_re = False

Thought: B's real part already fails to match, so B cannot be the answer regardless of what its imaginary part says; I move on to D. D's real part, $3$.
Action: Calculator
Action Input: test_D_re = Eq(re_val, 3)
Observation: test_D_re = True

Thought: D's real part matches; now D's imaginary part, $2$.
Action: Calculator
Action Input: test_D_im = Eq(im_val, 2)
Observation: test_D_im = True

Thought: Checking everything together against the four quotes copied at the start: A and C were already ruled out by the $i$-suffix format rule; B failed its real-part test; D passed both its real-part and imaginary-part tests, and D's own stated imaginary part never carried $i$ in the first place. D is the only option consistent on every count. That is the answer.
</think>

Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives $z=a+bi$ and asks for the real and imaginary parts of $\overline{z}$, among four options. Follow these steps, using the Calculator for every computation.

1. **Reading $z$.** Quote $z$ verbatim from the problem and store it.
2. **Computing the conjugate.** Compute `conj_z = conjugate(z)`. Never flip the sign of the imaginary part in your head; always let the Calculator do it, even though it looks simple.
3. **Reading off the real and imaginary parts.** Compute `re_val = re(conj_z)` and `im_val = im(conj_z)`.
4. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "...").
5. **Discarding any option whose stated imaginary part carries $i$, with no Calculator call.** A genuine imaginary part is always a bare real number; if an option's text (from the copy in step 4) shows its imaginary part written WITH $i$ attached (e.g. "$-2i$" instead of "$-2$"), that option is wrong by this format rule alone, and needs no numeric test at all. Only options whose imaginary part is written as a bare number remain candidates.
6. **Testing each remaining candidate's two numbers.** For each option still standing after step 5, in order: test its stated real part with `Eq(re_val, <that option's real-part number>)`; if that already comes back `False`, the option is eliminated and you move on without needing to test its imaginary part. If it comes back `True`, also test its stated imaginary part with `Eq(im_val, <that option's imaginary-part number>)`.
7. **Deciding.** Exactly one option should pass every check that applies to it (the $i$-suffix format rule in step 5, AND both numeric tests in step 6, whichever of those it reached). That option's letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

AN_SO_TU_DO = set()  # khong can an so tu do, chi doc va tinh truc tiep
MAX_TOOL_CALLS = 17          # quy trinh day du ~4 luot tinh (conj_z,re_val,im_val) + toi da 6 luot test 2 phuong an dung dinh dang, cong du cho vai lan sua loi cu phap

df_d04 = chay_mot_dang(
    ma_dang='D04',
    prompt_template=PROMPT_TEMPLATE,
    an_so_tu_do=AN_SO_TU_DO,
    max_tool_calls=MAX_TOOL_CALLS,
    data_path=f'{DATA_DIR}/plan_solve_prompts_D04.json',
    out_path=f'{OUT_DIR}/d04_react_calculator_full90.csv',
)


In [ ]:
# =====================================================================
# DANG D14 (prompt trich nguyen van tu KLTN_D14_ReAct_Calculator_1cau.ipynb)
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Adding the two complex numbers.** The problem always gives two complex numbers $z_1=a+bi$ and $z_2=c+di$ and asks for $|z_1+z_2|$. By fact 2, addition works componentwise: real part with real part, imaginary part with imaginary part. So $z_1+z_2=(a+c)+(b+d)i$, a single new complex number with real part $a+c$ and imaginary part $b+d$. This holds regardless of the sign of $a,b,c,d$ or whether they involve radicals; the rule never changes, only the arithmetic gets messier.

**Taking the modulus of the sum.** By fact 4, the modulus of any complex number $X+Yi$ is $\sqrt{X^2+Y^2}$. Applied here, with $X=a+c$ and $Y=b+d$ from the sum just found: $|z_1+z_2|=\sqrt{(a+c)^2+(b+d)^2}$. This single number is the entire answer the problem is asking for; there is no further simplification needed beyond evaluating this expression exactly, and no case analysis, since a modulus is always defined and always nonnegative regardless of what $a,b,c,d$ happen to be.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Cho hai số phức $z_1 = 1+i$ và $z_2 = 2-3i$. Tính môđun của số phức $z_1+z_2$.

A. $|z_1+z_2| = \sqrt{13}$.   B. $|z_1+z_2| = \sqrt{5}$.   C. $|z_1+z_1| = 1$.   D. $|z_1+z_2| = 5$.

<think>
Thought: The first number, quoted verbatim, is $z_1=1+i$. I store it.
Action: Calculator
Action Input: z1 = 1+I
Observation: z1 = 1 + I

Thought: The second number, quoted verbatim, is $z_2=2-3i$. I store it.
Action: Calculator
Action Input: z2 = 2-3*I
Observation: z2 = 2 - 3*I

Thought: By PART 2, add real parts with real parts and imaginary parts with imaginary parts.
Action: Calculator
Action Input: tong = z1+z2
Observation: tong = 3 - 2*I

Thought: Now the modulus of this sum; this single number is what every option is describing.
Action: Calculator
Action Input: modun = Abs(tong)
Observation: modun = sqrt(13)

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$|z_1+z_2| = \sqrt{13}$."; B: "$|z_1+z_2| = \sqrt{5}$."; C: "$|z_1+z_1| = 1$."; D: "$|z_1+z_2| = 5$." Regardless of exactly which letters they attach to inside the option text, each option states a single number to compare against `modun`; I read off that number for each and store all four together in one list, in order, so I never have to recall or retype any of them again.
Action: Calculator
Action Input: opts = [sqrt(13), sqrt(5), 1, 5]
Observation: opts = [sqrt(13), sqrt(5), 1, 5]

Thought: Now I test `modun` against each stored option by index, in order, even after a match is found, since this numeric check is not a formal proof. Option A is `opts[0]`.
Action: Calculator
Action Input: test_A = Eq(modun, opts[0])
Observation: test_A = True

Thought: Option B is `opts[1]`.
Action: Calculator
Action Input: test_B = Eq(modun, opts[1])
Observation: test_B = False

Thought: Option C is `opts[2]`.
Action: Calculator
Action Input: test_C = Eq(modun, opts[2])
Observation: test_C = False

Thought: Option D is `opts[3]`.
Action: Calculator
Action Input: test_D = Eq(modun, opts[3])
Observation: test_D = False

Thought: Checking every result against the quotes copied at the start: only `test_A` is `True`; `test_B`, `test_C`, `test_D` are all `False`. A is the only option consistent with `modun`. That is the answer.
</think>

Final Answer: \boxed{A}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives $z_1$ and $z_2$ and asks for $|z_1+z_2|$, among four options each stating a single number. Follow these steps, using the Calculator for every computation.

1. **Reading $z_1$ and $z_2$.** Quote both verbatim from the problem and store them.
2. **Adding them.** Compute `tong = z1+z2`.
3. **Taking the modulus.** Compute `modun = Abs(tong)`. This is exactly $|z_1+z_2|$, the value every option is describing.
4. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), then read off the single number each one states, and store all four numbers together in ONE Calculator call as a list: `opts = [<A>, <B>, <C>, <D>]`. This is the only time you read these numbers from the problem text; from here on, refer to each one only by index.
5. **Testing each option by index.** For each option A, B, C, D in turn: compare with `Eq(modun, opts[<matching index>])`. Test every option even after one already returns `True`; `Eq()` here is a numeric check, not a formal proof.
6. **Deciding.** Exactly one option should be `True`; that letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

AN_SO_TU_DO = set()  # khong can an so tu do, chi doc va tinh truc tiep
MAX_TOOL_CALLS = 17          # quy trinh day du ~4 luot tinh (tong,modun,opts) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap

df_d14 = chay_mot_dang(
    ma_dang='D14',
    prompt_template=PROMPT_TEMPLATE,
    an_so_tu_do=AN_SO_TU_DO,
    max_tool_calls=MAX_TOOL_CALLS,
    data_path=f'{DATA_DIR}/plan_solve_prompts_D14.json',
    out_path=f'{OUT_DIR}/d14_react_calculator_full90.csv',
)


In [ ]:
# =====================================================================
# DANG D28 (prompt trich nguyen van tu KLTN_D28_ReAct_Calculator_1cau.ipynb)
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Isolating $z$.** The problem always gives an equation of the shape $z\cdot w+(\text{free term})=(\text{right side})$, where $w$ is a fixed complex coefficient multiplying $z$. Moving every term that does not contain $z$ to the other side (ordinary algebra; this works for complex numbers exactly as it does for real ones) gives $z\cdot w=(\text{right side})-(\text{free term})$, and dividing both sides by $w$ isolates $z$ completely: $z=\dfrac{(\text{right side})-(\text{free term})}{w}$.

**Why the division simplifies to a clean $a+bi$ form.** By fact 8, dividing by a complex number $w$ is carried out by multiplying numerator and denominator by $w$'s conjugate $\overline{w}$; the denominator $w\overline{w}=|w|^2$ becomes a plain positive real number, which removes $i$ from the denominator entirely and leaves a genuine $a+bi$ form in the numerator. This is always possible and always exact, since $w$ is never zero in this problem type (a coefficient of zero would erase $z$ from the equation entirely, which never happens here).

**Reading off the modulus.** Once $z=a+bi$ is known exactly, fact 4 gives $|z|=\sqrt{a^2+b^2}$ directly; this single number is the entire answer, with no further case analysis needed.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Tính môđun của số phức $z$ thỏa mãn $z(2-i)+13i = 1$.

A. $|z| = \sqrt{34}$.   B. $|z| = 34$.   C. $|z| = \dfrac{5\sqrt{34}}{3}$.   D. $|z| = \dfrac{\sqrt{34}}{4}$.

<think>
Thought: The equation, quoted verbatim, is $z(2-i)+13i=1$. Moving the free term $13i$ to the right side gives $z(2-i)=1-13i$, so $z=\dfrac{1-13i}{2-i}$. I compute this division directly with the Calculator; it will carry out the conjugate-multiplication from PART 2 internally and return the exact $a+bi$ form.
Action: Calculator
Action Input: z = (1-13*I)/(2-I)
Observation: z = 3 - 5*I

Thought: Now the modulus of this $z$; this single number is what every option is describing.
Action: Calculator
Action Input: modun = Abs(z)
Observation: modun = sqrt(34)

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$|z| = \sqrt{34}$."; B: "$|z| = 34$."; C: "$|z| = \dfrac{5\sqrt{34}}{3}$."; D: "$|z| = \dfrac{\sqrt{34}}{4}$." Each states a single number to compare against `modun`; I read off that number for each and store all four together in one list, in order, so I never have to recall or retype any of them again.
Action: Calculator
Action Input: opts = [sqrt(34), 34, 5*sqrt(34)/3, sqrt(34)/4]
Observation: opts = [sqrt(34), 34, 5*sqrt(34)/3, sqrt(34)/4]

Thought: Now I test `modun` against each stored option by index, in order, even after a match is found, since this numeric check is not a formal proof. Option A is `opts[0]`.
Action: Calculator
Action Input: test_A = Eq(modun, opts[0])
Observation: test_A = True

Thought: Option B is `opts[1]`.
Action: Calculator
Action Input: test_B = Eq(modun, opts[1])
Observation: test_B = False

Thought: Option C is `opts[2]`.
Action: Calculator
Action Input: test_C = Eq(modun, opts[2])
Observation: test_C = False

Thought: Option D is `opts[3]`.
Action: Calculator
Action Input: test_D = Eq(modun, opts[3])
Observation: test_D = False

Thought: Checking every result against the quotes copied at the start: only `test_A` is `True`. A is the only option consistent with `modun`. That is the answer.
</think>

Final Answer: \boxed{A}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives an equation $z\cdot w+(\text{free term})=(\text{right side})$ and asks for $|z|$, among four options each stating a single number. Follow these steps, using the Calculator for every computation.

1. **Isolating and computing $z$.** Quote the equation verbatim. Rearrange it on paper (in your Thought, not with the Calculator) to $z=\dfrac{(\text{right side})-(\text{free term})}{w}$, keeping every sign exactly as printed, then compute that whole division in ONE Calculator call, e.g. `z = (1-13*I)/(2-I)`. The Calculator carries out the conjugate-multiplication and division exactly; never do this division by hand.
2. **Taking the modulus.** Compute `modun = Abs(z)`. This is exactly $|z|$, the value every option is describing.
3. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), then read off the single number each one states, and store all four numbers together in ONE Calculator call as a list: `opts = [<A>, <B>, <C>, <D>]`. This is the only time you read these numbers from the problem text; from here on, refer to each one only by index.
4. **Testing each option by index.** For each option A, B, C, D in turn: compare with `Eq(modun, opts[<matching index>])`. Test every option even after one already returns `True`; `Eq()` here is a numeric check, not a formal proof.
5. **Deciding.** Exactly one option should be `True`; that letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

AN_SO_TU_DO = set()  # khong can an so tu do, chi doc va tinh truc tiep
MAX_TOOL_CALLS = 17          # quy trinh day du ~3 luot tinh (z,modun,opts) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap

df_d28 = chay_mot_dang(
    ma_dang='D28',
    prompt_template=PROMPT_TEMPLATE,
    an_so_tu_do=AN_SO_TU_DO,
    max_tool_calls=MAX_TOOL_CALLS,
    data_path=f'{DATA_DIR}/plan_solve_prompts_D28.json',
    out_path=f'{OUT_DIR}/d28_react_calculator_full90.csv',
)


In [ ]:
# =====================================================================
# DANG D32 (prompt trich nguyen van tu KLTN_D32_ReAct_Calculator_1cau.ipynb)
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Algebraizing.** The problem always gives an equation mixing $z$ and $\overline{z}$ together (in this problem family, always exactly $3(\overline{z}-i)-(2+3i)z=(\text{right side})$), and asks for $|z|$. Write $z=x+yi$ so $\overline{z}=x-yi$ (facts 1, 3), and substitute both into the equation.

**Expanding to standard form.** Multiply out the left side term by term (fact 2), remembering $i^2=-1$ wherever it appears, and move the right side over so the whole equation reads (left side) $-$ (right side) $=0$. The result always has the shape $P+Qi=0$ for real polynomials $P,Q$ in $x,y$ (never any higher power than $1$, since neither $z$ nor $\overline{z}$ is ever squared in this problem type).

**Using equality of complex numbers.** By fact 1's uniqueness of the real/imaginary decomposition, $P+Qi=0$ forces BOTH $P=0$ and $Q=0$ separately (a complex number is zero exactly when its real and imaginary parts both are); $P$ is the real part of (left side) $-$ (right side), and $Q$ is the imaginary part.

**Solving the two linear equations, one at a time.** $P=0$ and $Q=0$ are two linear equations in $x,y$. Solve $P=0$ for $x$ in terms of $y$ first (or for $y$ in terms of $x$, whichever is more direct), then substitute that expression into $Q=0$ and solve the resulting single-variable equation to pin down the second unknown exactly; back-substitute to get the first. Never try to solve both equations simultaneously in one step; always one equation, one unknown, then substitute.

**Reading off the modulus.** Once $x,y$ are both known exactly, fact 4 gives $|z|=\sqrt{x^2+y^2}$ directly; this single number is the entire answer.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Cho số phức $z$ thoả mãn $3(\overline{z}-i)-(2+3i)z=7-16i$. Mô-đun của $z$ bằng

A. $\sqrt{5}$.   B. $5$.   C. $\sqrt{3}$.   D. $3$.

<think>
Thought: I algebraize $z$ per PART 2.
Action: Calculator
Action Input: z = x+y*I
Observation: z = x + I*y

Thought: And its conjugate.
Action: Calculator
Action Input: z_bar = conjugate(z)
Observation: z_bar = x - I*y

Thought: The equation, quoted verbatim, is $3(\overline{z}-i)-(2+3i)z=7-16i$. I expand the left side first.
Action: Calculator
Action Input: lhs = expand(3*(z_bar-I)-(2+3*I)*z)
Observation: lhs = x - 3*I*x + 3*y - 5*I*y - 3*I

Thought: Now move the right side over, so the whole equation reads as an expression equal to $0$.
Action: Calculator
Action Input: eq_expr = lhs - (7-16*I)
Observation: eq_expr = x - 3*I*x + 3*y - 5*I*y - 7 + 13*I

Thought: By PART 2, this being $0$ splits into its real part being $0$ and its imaginary part being $0$, separately. The real part first.
Action: Calculator
Action Input: real_p = re(eq_expr)
Observation: real_p = x + 3*y - 7

Thought: And the imaginary part.
Action: Calculator
Action Input: imag_p = im(eq_expr)
Observation: imag_p = -3*x - 5*y + 13

Thought: I solve `real_p = 0` for $x$ first, in terms of $y$; never both equations at once.
Action: Calculator
Action Input: x_sols = solve(Eq(real_p, 0), x)
Observation: x_sols = [7 - 3*y]

Thought: Exactly one expression, as expected for a linear equation. I store it by index.
Action: Calculator
Action Input: x_expr = x_sols[0]
Observation: x_expr = 7 - 3*y

Thought: Now substitute this into `imag_p`, to get a single equation purely in $y$.
Action: Calculator
Action Input: imag_sub = imag_p.subs(x, x_expr)
Observation: imag_sub = 4*y - 8

Thought: Solve this for $y$.
Action: Calculator
Action Input: y_sols = solve(Eq(imag_sub, 0), y)
Observation: y_sols = [2]

Thought: Exactly one value, as expected. I store it by index.
Action: Calculator
Action Input: y_val = y_sols[0]
Observation: y_val = 2

Thought: Back-substitute into `x_expr` to get the matching $x$.
Action: Calculator
Action Input: x_val = x_expr.subs(y, y_val)
Observation: x_val = 1

Thought: Now the modulus; this single number is what every option is describing.
Action: Calculator
Action Input: modun = sqrt(x_val**2 + y_val**2)
Observation: modun = sqrt(5)

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$\sqrt{5}$."; B: "$5$."; C: "$\sqrt{3}$."; D: "$3$." I store all four together in one list, in order, so I never have to recall or retype any of them again.
Action: Calculator
Action Input: opts = [sqrt(5), 5, sqrt(3), 3]
Observation: opts = [sqrt(5), 5, sqrt(3), 3]

Thought: Now I test `modun` against each stored option by index, in order, even after a match is found. Option A is `opts[0]`.
Action: Calculator
Action Input: test_A = Eq(modun, opts[0])
Observation: test_A = True

Thought: Option B is `opts[1]`.
Action: Calculator
Action Input: test_B = Eq(modun, opts[1])
Observation: test_B = False

Thought: Option C is `opts[2]`.
Action: Calculator
Action Input: test_C = Eq(modun, opts[2])
Observation: test_C = False

Thought: Option D is `opts[3]`.
Action: Calculator
Action Input: test_D = Eq(modun, opts[3])
Observation: test_D = False

Thought: Checking every result against the quotes copied at the start: only `test_A` is `True`. A is the only option consistent with `modun`. That is the answer.
</think>

Final Answer: \boxed{A}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives $3(\overline{z}-i)-(2+3i)z=(\text{right side})$ and asks for $|z|$, among four options each stating a single number. Follow these steps, using the Calculator for every computation.

1. **Algebraizing.** Compute `z = x+y*I` and `z_bar = conjugate(z)`.
2. **Expanding the equation.** Quote the equation verbatim and compute `lhs = expand(3*(z_bar-I)-(2+3*I)*z)`, then `eq_expr = lhs - (right side)` (with the right side read exactly as printed, keeping every sign).
3. **Splitting into real and imaginary parts.** Compute `real_p = re(eq_expr)` and `imag_p = im(eq_expr)`. By PART 2, both must equal $0$.
4. **Solving one equation, one unknown, at a time.** Never call `solve()` with a list of both equations and both unknowns at once. Instead: compute `x_sols = solve(Eq(real_p, 0), x)`, store `x_expr = x_sols[0]` (indexing, never retyping). Substitute: `imag_sub = imag_p.subs(x, x_expr)`. Then `y_sols = solve(Eq(imag_sub, 0), y)`, store `y_val = y_sols[0]`. Back-substitute: `x_val = x_expr.subs(y, y_val)`.
5. **The modulus.** Compute `modun = sqrt(x_val**2 + y_val**2)`.
6. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), then read off the single number each one states, and store all four together in ONE Calculator call as a list: `opts = [<A>, <B>, <C>, <D>]`.
7. **Testing each option by index.** For each option A, B, C, D in turn: compare with `Eq(modun, opts[<matching index>])`. Test every option even after one already returns `True`.
8. **Deciding.** Exactly one option should be `True`; that letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

AN_SO_TU_DO = {'x', 'y'}  # x,y: phan thuc/ao cua z=x+yi
MAX_TOOL_CALLS = 27          # quy trinh day du ~13 luot tinh (lhs,eq_expr,real_p,imag_p,x_sols,x_expr,imag_sub,y_sols,y_val,x_val,modun,opts) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap

df_d32 = chay_mot_dang(
    ma_dang='D32',
    prompt_template=PROMPT_TEMPLATE,
    an_so_tu_do=AN_SO_TU_DO,
    max_tool_calls=MAX_TOOL_CALLS,
    data_path=f'{DATA_DIR}/plan_solve_prompts_D32.json',
    out_path=f'{OUT_DIR}/d32_react_calculator_full90.csv',
)


In [ ]:
# =====================================================================
# DANG D33 (prompt trich nguyen van tu KLTN_D33_ReAct_Calculator_1cau.ipynb)
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Algebraizing.** The problem always gives $z=a+bi$ ($a,b$ real, these are exactly the unknowns to find) satisfying $z+(\text{free term})-|z|i=0$, and asks for $S=a+kb$ for some fixed real $k$. By fact 4, $|z|=\sqrt{a^2+b^2}$; substituting this and $z=a+bi$ turns the whole condition into an equation purely in the real unknowns $a,b$: $a+bi+(\text{free term})-\sqrt{a^2+b^2}\,i=0$.

**Splitting into real and imaginary parts.** Write the free term as $c+di$ ($c,d$ real, read directly off the problem). Grouping everything without $i$ together and everything with $i$ together gives $(a+c)+(b+d-\sqrt{a^2+b^2})i=0$. By fact 1's uniqueness of the decomposition, this forces BOTH the real part $a+c=0$ AND the imaginary part $b+d-\sqrt{a^2+b^2}=0$.

**Solving for $a$ first.** The real-part equation $a+c=0$ involves only $a$; it gives $a=-c$ immediately, with no need to touch the imaginary-part equation yet.

**Solving for $b$ next, substituting $a$ in.** With $a$ now a known number, the imaginary-part equation $b+d=\sqrt{a^2+b^2}$ is a single equation in the single unknown $b$ (it still has a square root in it, but that is fine: `solve(...)` handles a radical equation in one unknown directly, finding exactly the value or values of $b$ that make it hold, without needing to square both sides by hand first).

**Reading off $S$.** Once $a$ and $b$ are both known exactly, $S=a+kb$ (with $k$ read directly from the problem's own definition of $S$) is a single number: the entire answer.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Cho số phức $z=a+bi$ ($a,b \in \mathbb{R}$) thỏa mãn $z+1+3i-|z|i=0$. Tính $S=a+3b$.

A. $S=\dfrac{7}{3}$.   B. $S=-5$.   C. $S=5$.   D. $S=-\dfrac{7}{3}$.

<think>
Thought: The free term added to $z$, quoted verbatim from "$z+1+3i-|z|i=0$", is $1+3i$. I store it.
Action: Calculator
Action Input: const = 1+3*I
Observation: const = 1 + 3*I

Thought: Its real part, $c$.
Action: Calculator
Action Input: c_val = re(const)
Observation: c_val = 1

Thought: And its imaginary part, $d$.
Action: Calculator
Action Input: d_val = im(const)
Observation: d_val = 3

Thought: By PART 2, the real-part equation is $a+c=0$; I solve it for $a$ alone, before touching the imaginary part at all.
Action: Calculator
Action Input: a_sols = solve(Eq(a+c_val, 0), a)
Observation: a_sols = [-1]

Thought: Exactly one solution, as expected for a linear equation. I store it by index.
Action: Calculator
Action Input: a_val = a_sols[0]
Observation: a_val = -1

Thought: Now the imaginary-part equation, $b+d=\sqrt{a^2+b^2}$, with $a$ substituted in; I solve it directly for $b$, radical and all.
Action: Calculator
Action Input: b_sols = solve(Eq(b+d_val, sqrt(a_val**2+b**2)), b)
Observation: b_sols = [-4/3]

Thought: Exactly one solution here too. I store it by index.
Action: Calculator
Action Input: b_val = b_sols[0]
Observation: b_val = -4/3

Thought: The problem defines $S=a+3b$, quoted verbatim; I compute it now that both $a$ and $b$ are known.
Action: Calculator
Action Input: S = a_val + 3*b_val
Observation: S = -5

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$S=\dfrac{7}{3}$."; B: "$S=-5$."; C: "$S=5$."; D: "$S=-\dfrac{7}{3}$." I store all four together in one list, in order, so I never have to recall or retype any of them again.
Action: Calculator
Action Input: opts = [7/3, -5, 5, -7/3]
Observation: opts = [7/3, -5, 5, -7/3]

Thought: Now I test `S` against each stored option by index, in order, even after a match is found. Option A is `opts[0]`.
Action: Calculator
Action Input: test_A = Eq(S, opts[0])
Observation: test_A = False

Thought: Option B is `opts[1]`.
Action: Calculator
Action Input: test_B = Eq(S, opts[1])
Observation: test_B = True

Thought: Option C is `opts[2]`.
Action: Calculator
Action Input: test_C = Eq(S, opts[2])
Observation: test_C = False

Thought: Option D is `opts[3]`.
Action: Calculator
Action Input: test_D = Eq(S, opts[3])
Observation: test_D = False

Thought: Checking every result against the quotes copied at the start: only `test_B` is `True`. B is the only option consistent with `S`. That is the answer.
</think>

Final Answer: \boxed{B}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives $z=a+bi$ satisfying $z+(\text{free term})-|z|i=0$ and asks for $S=a+kb$ (some fixed $k$), among four options each stating a single number. Follow these steps, using the Calculator for every computation.

1. **Reading the free term.** Quote it verbatim from the equation and store it, then compute `c_val = re(const)` and `d_val = im(const)`.
2. **Solving for $a$.** Compute `a_sols = solve(Eq(a+c_val, 0), a)`, store `a_val = a_sols[0]` (indexing, never retyping). This uses only the real part; do not touch the imaginary-part equation yet.
3. **Solving for $b$.** Compute `b_sols = solve(Eq(b+d_val, sqrt(a_val**2+b**2)), b)`, store `b_val = b_sols[0]`. `solve(...)` handles the square root directly; never square both sides by hand yourself.
4. **Computing $S$.** Quote the problem's own definition of $S$ verbatim (read off the coefficient $k$ exactly as printed, including its sign) and compute `S = a_val + k*b_val`.
5. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), then read off the single number each one states, and store all four together in ONE Calculator call as a list: `opts = [<A>, <B>, <C>, <D>]`.
6. **Testing each option by index.** For each option A, B, C, D in turn: compare with `Eq(S, opts[<matching index>])`. Test every option even after one already returns `True`.
7. **Deciding.** Exactly one option should be `True`; that letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

AN_SO_TU_DO = {'a', 'b'}  # a,b: phan thuc/ao cua z=a+bi (an so can giai)
MAX_TOOL_CALLS = 23          # quy trinh day du ~9 luot tinh (const,c_val,d_val,a_sols,a_val,b_sols,b_val,S,opts) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap

df_d33 = chay_mot_dang(
    ma_dang='D33',
    prompt_template=PROMPT_TEMPLATE,
    an_so_tu_do=AN_SO_TU_DO,
    max_tool_calls=MAX_TOOL_CALLS,
    data_path=f'{DATA_DIR}/plan_solve_prompts_D33.json',
    out_path=f'{OUT_DIR}/d33_react_calculator_full90.csv',
)
